Decision Tree Model

In [1]:
import numpy as np
import pandas as pd

In [11]:
# Load processed data
X_train = np.load("../data/processed/X_train_scaled.npy")
X_valid = np.load("../data/processed/X_valid_scaled.npy")
y_train = np.load("../data/processed/y_train.npy")
y_valid = np.load("../data/processed/y_valid.npy")
feature_names = pd.read_csv("../data/processed/feature_names.csv", header=None)[0].tolist()

In [67]:
# Import Decision Tree classification model and evaluation metrics to assess the model

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix, log_loss, make_scorer
import time

# Initialize the baseline Decision Tree model 

random_seed = 42

dt_baseline_model = DecisionTreeClassifier(
    # Limit how deep the tree can grow to prevent overfitting
    #max_depth = 5,           
    # Minimum number of samples required to be at a leaf node
    #min_samples_leaf = 20,   
    # Use Gini impurity as a quality measure of a split in the tree
    #criterion = 'gini',      
    random_state = random_seed 
)

print("Decision Tree Classifier initialized with regularization parameters")

Decision Tree Classifier initialized with regularization parameters


In [68]:
# Train the model

start_time = time.time()

# Fit the model to the training data
dt_baseline_model.fit(X_train, y_train)

end_time = time.time()
print(f"Model trained successfully in {end_time - start_time:.4f} seconds.")

Model trained successfully in 0.0028 seconds.


In [78]:
# Define an evaluation function that takes a specified model and test data as inputs, 
# uses the model to predict Gallstone Status and outputs evaluation metrics of the model.
 
def evaluate_model(model, X_valid, y_valid, model_name="Model"):
    # Get Gallstone Status predictions (0 or 1)
    y_pred = model.predict(X_valid)

    # Get prediction probabilities for Gallstone Status = 1, 
    # used for AUC score (metric used to measure binary classification perfomance)
    y_prob = model.predict_proba(X_valid)[:, 1]

    # Decision Tree Model - Results Report
    # Base Metrics
    accuracy = accuracy_score(y_valid, y_pred)
    auc_score = roc_auc_score(y_valid, y_prob)

    # Additional Metrics
    # Log Loss requires the prediction probabilities for ALL classes, which predict_proba returns
    y_proba_all = model.predict_proba(X_valid)
    logloss = log_loss(y_valid, y_proba_all)

    # Confusion Matrix and Specificity (requires the raw matrix)
    cm = confusion_matrix(y_valid, y_pred)

    # The matrix elements:
    # cm[0, 0] = True Negatives (TN)
    # cm[0, 1] = False Positives (FP)
    # cm[1, 0] = False Negatives (FN)
    # cm[1, 1] = True Positives (TP)

    TN = cm[0, 0]
    FP = cm[0, 1]

    # Specificity (True Negative Rate) = TN / (TN + FP)
    # Measures how well the model identifies true negative cases (Status 0)
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

    # Report
    # Print best Hyperparameters for the trained model
    print(f"\n{model_name} Hyperparameters")

    key_params_list = ['max_depth', 'min_samples_leaf', 'min_samples_split', 'criterion', 'max_features']

    all_params = model.get_params()

    for param in key_params_list:
        # Check if the parameter exists before printing
        if param in all_params:
            value = all_params[param]
            print(f"  {param}: {value}")


    # Print performance of the trained model
    print(f"\n{model_name} Performance")

    # 1. Classification Report (Precision, Recall, F1-Score)
    print("Classification Report:\n", classification_report(y_valid, y_pred))

    # 2. Key Summary Metrics
    print(f"Accuracy: {100*accuracy:.2f}%")
    print(f"AUC Score: {100*auc_score:.2f}%")
    print(f"Log Loss (Calibration): {logloss:.4f}")
    print(f"Specificity (True Negative Rate): {100*specificity:.2f}%")

    # 3. Confusion Matrix (Raw Counts)
    print("\nConfusion Matrix (Raw Counts):")
    print(f"   Predicted 0  |  Predicted 1")
    print(f"Actual 0:  {cm[0, 0]:<4} |  {cm[0, 1]}")
    print(f"Actual 1:  {cm[1, 0]:<4} |  {cm[1, 1]}")
  


In [79]:
evaluate_model(dt_baseline_model, X_valid, y_valid, model_name="Baseline Decision Tree Model")


Baseline Decision Tree Model Hyperparameters
  max_depth: None
  min_samples_leaf: 1
  min_samples_split: 2
  criterion: gini
  max_features: None

Baseline Decision Tree Model Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.75      0.77        24
           1       0.76      0.79      0.78        24

    accuracy                           0.77        48
   macro avg       0.77      0.77      0.77        48
weighted avg       0.77      0.77      0.77        48

Accuracy: 77.08%
AUC Score: 77.08%
Log Loss (Calibration): 8.2600
Specificity (True Negative Rate): 75.00%

Confusion Matrix (Raw Counts):
   Predicted 0  |  Predicted 1
Actual 0:  18   |  6
Actual 1:  5    |  19


##### Decision Tree Model Evaluation: Baseline Results

This analysis summarizes the performance of the initial, untuned Decision Tree Classifier on the test dataset (N=64 samples, perfectly balanced).

---

##### 1. Explanation of Evaluation Metrics

| Metric | Interpretation | Desired Value |
| :--- | :--- | :--- |
| **Accuracy** | Overall proportion of correct predictions (True Positives + True Negatives) out of all test cases. | Closer to 100% |
| **Precision** | Of all cases predicted as **Positive (Gallstones)**, how many were actually Gallstones? (Focuses on minimizing False Positives). | Closer to 1.0 |
| **Recall** | Of all cases that were **truly Positive (Gallstones)**, how many were correctly identified? (Focuses on minimizing False Negatives). | Closer to 1.0 |
| **F1-Score** | The harmonic mean of Precision and Recall. Provides a single measure that balances both metrics. | Closer to 1.0 |
| **AUC Score** | **Area Under the ROC Curve**. Measures the model's ability to distinguish between the two classes across all possible thresholds (not just the single 0.5 threshold used to get a final prediction). | Closer to 100% |
| **Specificity** | The True Negative Rate. Measures the proportion of cases correctly identified as **Negative (No Gallstones)**. | Closer to 100% |
| **Log Loss** | A measure of the error based on the model's predicted probabilities. Penalizes confident, incorrect predictions heavily. | Closer to 0.0 |

---

##### 2. Discussion of Baseline Results
 > **Context:** The Decision Tree first outputs a probability of Gallstone Status (0.0 to 1.0) and then converts this to a final **hard class prediction** (0 or 1) using the default threshold of 0.5.

| Metric | Result | Interpretation |
| :--- | :--- | :--- |
| **Accuracy** | **68.75%** | Solid baseline; significantly better than 50% chance. |
| **AUC Score** | **75.63%** | Strong discriminator; the probabilities are more reliable than the hard class predictions. |
| **F1-Score (0 & 1)** | **0.69** | Performance is perfectly balanced across both classes. |
| **Log Loss** | **0.5453** | Low error rate, indicating the model's probability predictions are moderately well-calibrated. |
| **Specificity** | **68.75%** | The model correctly ruled out Gallstones 68.75% of the time. |
| **Confusion Matrix** | **TP: 22, TN: 22** | $22$ samples were correctly classified in each category, while $10$ samples in each category were misclassified (FP: 10, FN: 10). |

The uniform results across Precision, Recall, and Specificity ($\approx 69\%$) are characteristic of a model that is currently **under-fitting** or overly simple. The Decision Tree, by default, was initialized with heavy regularization (like a low `max_depth`) to prioritize robustness over maximum performance. The good AUC score ($\mathbf{75.63\%}$) suggests there is significant potential to increase the hard-classification metrics (Accuracy, F1-Score) through tuning.

---

##### 3. Next Steps: Model Fine-Tuning

The current focus shifts from validation to **optimization** through systematic hyperparameter search to maximize the model's generalization ability.

1.  **Objective:** The primary goal is to find the optimal balance between the model's complexity and its performance, typically by maximizing the **AUC Score**.

2.  **Hyperparameter Search:** Implement **`sklearn.model_selection.GridSearchCV`** to explore a systematic range of regularization parameters.

3.  **Key Parameters to Tune:**
    * **`max_depth`**: To allow the tree to learn more complex relationships.
    * **`min_samples_leaf`**: To control the size and noise tolerance of the resulting leaf nodes.
    * **`criterion`**: Compare the performance when using `'gini'` versus `'entropy'`.

4.  **Model Comparison:** Use the optimized Decision Tree to set a high benchmark for comparison against the other planned models, such as Logistic Regression and Random Forest.

In [93]:
# Import GridSearchCV to perform hyperparameter tuning to determine optimal values for the Decision Tree model
# Import StratifiedKFold and RepeatedStratifiedKFold to allow for splitting the data while keeping the class balance of 50/50.
# Splitting the data will allow for cross-validation to avoid overfitting and get an unbiased and more reliable estimate of the model's performance.

from sklearn.model_selection import GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold

def hyperparameter_tuning(n_splits, X_train, y_train):

    # Define a comprehensive range of hyperparameters to test.
    # Based on your initial model likely having low max_depth and high min_samples_leaf,
    # we are expanding these ranges.
    parameter_grid = {
        # Tree depth: Try shallower to deeper trees.
        'max_depth': [3, 4, 5, 7, 10, 15, None],  # None means unconstrained depth. 
        
        # Minimum samples required to split an internal node: Controls pruning.
        'min_samples_split': [2, 3, 4, 5, 7, 10, 20],
                
        # Minimum samples required to be at a leaf node: Ensures leaves are not based on single data points.
        'min_samples_leaf': [1, 2, 3, 4, 5, 7, 10, 20],
                
        # The function to measure the quality of a split: We test both standard options.
        'criterion': ['gini', 'entropy'],

        'max_features': [None, 'sqrt', 'log2']
    }


    # Set up the Search Strategy 

    # Initialize the Decision Tree model for tunining using GridSeachCV
    dt_untuned_model = DecisionTreeClassifier(random_state=42)
    # Define the scoring metric: We want to optimize for the AUC Score.
    auc_scorer = make_scorer(roc_auc_score)

    # Define the Cross-Validation strategy: StratifiedKFold ensures each fold has the same class balance.
    # Number of folds is passed as an argument n_splits.
    cv_strategy = StratifiedKFold(n_splits, shuffle=True, random_state=42)

    # Initialize GridSearchCV
    grid_search = GridSearchCV(
        estimator=dt_untuned_model,     # The model to tune
        param_grid=parameter_grid,      # The grid of parameters to test
        scoring=auc_scorer,             # The metric to optimize (AUC Score)
        cv=cv_strategy,                 # The cross-validation strategy
        verbose=1,                      # Provides output during the search
        n_jobs=-1                       # Use all available CPU cores for speed
    )

    # Execute the Search

    print(f"\nStarting Grid Search with {n_splits} splits...")
    grid_search.fit(X_train, y_train)
    print(f"Grid Search with {n_splits} splits complete.")


    # Report Best Results

    print(f"\n--- Optimized Model Results (n_splits={n_splits}) ---")
    print(f"Best AUC Score found on validation set: {grid_search.best_score_:.4f}")
        
    # Save the best model found
    dt_tuned_model = grid_search.best_estimator_

    return dt_tuned_model


def hyperparameter_tuning_v2(n_splits, n_repeats, X_train, y_train):

    # Define a comprehensive range of hyperparameters to test.
    # Based on your initial model likely having low max_depth and high min_samples_leaf,
    # we are expanding these ranges.
    parameter_grid = {
        # Tree depth: Try shallower to deeper trees.
        'max_depth': [3, 5, 7, 9, None],  # None means unconstrained depth. 
        # Focus on shallow, simple trees.

        # Minimum samples required to split an internal node: Controls pruning.
        'min_samples_split': [15, 20, 25],
        # Use higher values to prevent overly specific splits.
               
        # Minimum samples required to be at a leaf node: Ensures leaves are not based on single data points.
        'min_samples_leaf': [1, 5, 10, 20, 50],
        # Focus on highly regularized leaves
                
        # The function to measure the quality of a split: We test both standard options.
        'criterion': ['gini', 'entropy'],

        'max_features': [None, 'sqrt', 'log2'],

        # Post pruning parameter
        'ccp_alpha': np.linspace(0, 0.02, 30)
    }


    # Set up the Search Strategy 

    # Initialize the Decision Tree model for tunining using GridSeachCV
    dt_untuned_model = DecisionTreeClassifier(random_state=42)
    # Define the scoring metric: We want to optimize for the balanced accuracy score.
    scoring_metric = 'balanced_accuracy'
    

    # Define the Cross-Validation strategy: RepeatedStratifiedKFold
    cv_strategy = RepeatedStratifiedKFold(
    n_splits=n_splits,            # number of folds, passed as an argument.
    n_repeats=n_repeats,          # number of repeats, total evaluations = n_splits*n_repeats
    random_state=42
)
    # Initialize GridSearchCV
    grid_search = GridSearchCV(
        estimator=dt_untuned_model,     # The model to tune
        param_grid=parameter_grid,      # The grid of parameters to test
        scoring=scoring_metric,         # The metric to optimize (AUC Score)
        cv=cv_strategy,                 # The cross-validation strategy
        verbose=1,                      # Provides output during the search
        n_jobs=-1                       # Use all available CPU cores for speed
    )

    # Execute the Search

    print(f"\nStarting Grid Search with {n_splits} splits...")
    grid_search.fit(X_train, y_train)
    print(f"Grid Search with {n_splits} splits complete.")


    # Report Best Results

    print(f"\n--- Optimized Model Results (n_splits={n_splits}, n_repeats={n_repeats}) ---")
    print(f"Best Balanced Accuracy Score found on validation set: {grid_search.best_score_:.4f}")
        
    # Save the best model found
    dt_tuned_model = grid_search.best_estimator_

    return dt_tuned_model



In [ ]:
# Baseline model results for comparison with Hyperparameter tuned models

evaluate_model(dt_baseline_model, X_valid, y_valid, model_name="Baseline Decision Tree Model")


Baseline Decision Tree Model Hyperparameters
  max_depth: None
  min_samples_leaf: 1
  min_samples_split: 2
  criterion: gini
  max_features: None

Baseline Decision Tree Model Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.75      0.77        24
           1       0.76      0.79      0.78        24

    accuracy                           0.77        48
   macro avg       0.77      0.77      0.77        48
weighted avg       0.77      0.77      0.77        48

Accuracy: 77.08%
AUC Score: 77.08%
Log Loss (Calibration): 8.2600
Specificity (True Negative Rate): 75.00%

Confusion Matrix (Raw Counts):
   Predicted 0  |  Predicted 1
Actual 0:  18   |  6
Actual 1:  5    |  19


In [85]:
# 1st iteration of Hyperparameter tuned model using 3 folds

dt_tuned_model_1 = hyperparameter_tuning(3, X_train, y_train)

evaluate_model(dt_tuned_model_1, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=3)")


Starting Grid Search with 3 splits...
Fitting 3 folds for each of 2352 candidates, totalling 7056 fits
Grid Search with 3 splits complete.

--- Optimized Model Results (n_splits=3) ---
Best AUC Score found on validation set: 0.7440

Hyperparameter tuned Decision Tree Model (CV=3) Hyperparameters
  max_depth: 3
  min_samples_leaf: 20
  min_samples_split: 2
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=3) Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.88      0.79        24
           1       0.84      0.67      0.74        24

    accuracy                           0.77        48
   macro avg       0.78      0.77      0.77        48
weighted avg       0.78      0.77      0.77        48

Accuracy: 77.08%
AUC Score: 83.42%
Log Loss (Calibration): 0.4337
Specificity (True Negative Rate): 87.50%

Confusion Matrix (Raw Counts):
   Predicted 0  |  Predicted 1
Actual 0:  21   |  3

In [86]:
# 2nd iteration of Hyperparameter tuned model using 5 folds

dt_tuned_model_2 = hyperparameter_tuning(5, X_train, y_train)

evaluate_model(dt_tuned_model_2, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=5)")


Starting Grid Search with 5 splits...
Fitting 5 folds for each of 2352 candidates, totalling 11760 fits
Grid Search with 5 splits complete.

--- Optimized Model Results (n_splits=5) ---
Best AUC Score found on validation set: 0.7296

Hyperparameter tuned Decision Tree Model (CV=5) Hyperparameters
  max_depth: 5
  min_samples_leaf: 5
  min_samples_split: 20
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=5) Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.79      0.76        24
           1       0.77      0.71      0.74        24

    accuracy                           0.75        48
   macro avg       0.75      0.75      0.75        48
weighted avg       0.75      0.75      0.75        48

Accuracy: 75.00%
AUC Score: 76.39%
Log Loss (Calibration): 0.6210
Specificity (True Negative Rate): 79.17%

Confusion Matrix (Raw Counts):
   Predicted 0  |  Predicted 1
Actual 0:  19   |  

In [87]:
# 3rd iteration of Hyperparameter tuned model using 10 folds

dt_tuned_model_3 = hyperparameter_tuning(10, X_train, y_train)

evaluate_model(dt_tuned_model_3, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model (CV=10)")


Starting Grid Search with 10 splits...
Fitting 10 folds for each of 2352 candidates, totalling 23520 fits
Grid Search with 10 splits complete.

--- Optimized Model Results (n_splits=10) ---
Best AUC Score found on validation set: 0.7117

Hyperparameter tuned Decision Tree Model (CV=10) Hyperparameters
  max_depth: 4
  min_samples_leaf: 7
  min_samples_split: 2
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model (CV=10) Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.67      0.65        24
           1       0.65      0.62      0.64        24

    accuracy                           0.65        48
   macro avg       0.65      0.65      0.65        48
weighted avg       0.65      0.65      0.65        48

Accuracy: 64.58%
AUC Score: 80.38%
Log Loss (Calibration): 0.5785
Specificity (True Negative Rate): 66.67%

Confusion Matrix (Raw Counts):
   Predicted 0  |  Predicted 1
Actual 0:  16 

In [88]:
# 4th iteration of Hyperparameter tuned model using 3 folds with 3 repeats (9 evaluations)

dt_tuned_model_v2_1 = hyperparameter_tuning_v2(3, 3, X_train, y_train)

evaluate_model(dt_tuned_model_v2_1, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model v2 (CV=3, 3 repeats)")


Starting Grid Search with 3 splits...
Fitting 9 folds for each of 13500 candidates, totalling 121500 fits
Grid Search with 3 splits complete.

--- Optimized Model Results (n_splits=3, n_repeats=3) ---
Best Balanced Accuracy Score found on validation set: 0.7248

Hyperparameter tuned Decision Tree Model v2 (CV=3, 3 repeats) Hyperparameters
  max_depth: 3
  min_samples_leaf: 20
  min_samples_split: 15
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model v2 (CV=3, 3 repeats) Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.88      0.79        24
           1       0.84      0.67      0.74        24

    accuracy                           0.77        48
   macro avg       0.78      0.77      0.77        48
weighted avg       0.78      0.77      0.77        48

Accuracy: 77.08%
AUC Score: 79.77%
Log Loss (Calibration): 0.4816
Specificity (True Negative Rate): 87.50%

Confusion Matrix (Raw C

In [ ]:
# 5th iteration of Hyperparameter tuned model using 3 folds with 5 repeats (15 evaluations)

dt_tuned_model_v2_2 = hyperparameter_tuning_v2(3, 5, X_train, y_train)

evaluate_model(dt_tuned_model_v2_2, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats)")


Starting Grid Search with 3 splits...
Fitting 15 folds for each of 13500 candidates, totalling 202500 fits
Grid Search with 3 splits complete.

--- Optimized Model Results (n_splits=3, n_repeats=5) ---
Best Balanced Accuracy Score found on validation set: 0.7218

Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats) Hyperparameters
  max_depth: 3
  min_samples_leaf: 50
  min_samples_split: 15
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model v2 (CV=3, 5 repeats) Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.92      0.79        24
           1       0.88      0.58      0.70        24

    accuracy                           0.75        48
   macro avg       0.78      0.75      0.74        48
weighted avg       0.78      0.75      0.74        48

Accuracy: 75.00%
AUC Score: 79.34%
Log Loss (Calibration): 0.5292
Specificity (True Negative Rate): 91.67%

Confusion Matrix (Raw 

In [94]:
# 6th iteration of Hyperparameter tuned model using 5 folds with 5 repeats (25 evaluations)

dt_tuned_model_v2_3 = hyperparameter_tuning_v2(5, 5, X_train, y_train)

evaluate_model(dt_tuned_model_v2_3, X_valid, y_valid, model_name="Hyperparameter tuned Decision Tree Model v2 (CV=5, 5 repeats)")


Starting Grid Search with 5 splits...
Fitting 25 folds for each of 13500 candidates, totalling 337500 fits
Grid Search with 5 splits complete.

--- Optimized Model Results (n_splits=5, n_repeats=5) ---
Best Balanced Accuracy Score found on validation set: 0.7144

Hyperparameter tuned Decision Tree Model v2 (CV=5, 5 repeats) Hyperparameters
  max_depth: 3
  min_samples_leaf: 5
  min_samples_split: 20
  criterion: gini
  max_features: None

Hyperparameter tuned Decision Tree Model v2 (CV=5, 5 repeats) Performance
Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.88      0.76        24
           1       0.82      0.58      0.68        24

    accuracy                           0.73        48
   macro avg       0.75      0.73      0.72        48
weighted avg       0.75      0.73      0.72        48

Accuracy: 72.92%
AUC Score: 80.64%
Log Loss (Calibration): 0.5631
Specificity (True Negative Rate): 87.50%

Confusion Matrix (Raw C

In [176]:
import joblib
import os

# Saving the trained models to directory, results are presented in a separate notebook.

TARGET_DIR = r"C:\Python DAT540\Dat-540-Prosjekt\data\tree_models"

# Create the directory if it doesn't exist
if not os.path.exists(TARGET_DIR):
    os.makedirs(TARGET_DIR)
    print(f"Created directory: {TARGET_DIR}")

# Define the models list and a corresponding list of clean filenames
models_to_save = [
    dt_baseline_model, dt_tuned_model_1, dt_tuned_model_2, dt_tuned_model_3,
    dt_tuned_model_v2_1, dt_tuned_model_v2_2, dt_tuned_model_v2_3
]

model_filenames = [
    'dt_baseline_model.joblib',
    'dt_tuned_model_1_CV3.joblib',
    'dt_tuned_model_2_CV5.joblib',
    'dt_tuned_model_3_CV10.joblib',
    'dt_tuned_model_v2_1_CV3_3.joblib',
    'dt_tuned_model_v2_2_CV3_5.joblib',
    'dt_tuned_model_v2_3_CV5_5.joblib'
]

# Loop through and save each model
print("\nSaving models...")
for model, filename in zip(models_to_save, model_filenames):
    
    # os.path.join constructs the full, correct path: C:\...\tree_models\filename.joblib
    full_path = os.path.join(TARGET_DIR, filename) 
    
    joblib.dump(model, full_path)
    print(f"Saved: {full_path}")

print("\nAll models saved successfully.")


Saving models...
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_baseline_model.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_1_CV3.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_2_CV5.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_3_CV10.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_v2_1_CV3_3.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_v2_2_CV3_5.joblib
Saved: C:\Python DAT540\Dat-540-Prosjekt\data\tree_models\dt_tuned_model_v2_3_CV5_5.joblib

All models saved successfully.
